# Query the local ABC-130k catalog

Example queries against the locally registered `abc_130k` dataset.
Register it first — see the README's "Local Catalog" section:

```bash
pixi run serve          # in one shell, leave running
pixi run abc-register   # in another shell
```

In [ ]:
# import packages and connect to the catalog
from __future__ import annotations

from collections import Counter

import pandas as pd
from rerun.catalog import CatalogClient

CATALOG_URL = "rerun+http://127.0.0.1:51234"
DATASET_NAME = "abc_130k"

try:
    client = CatalogClient(CATALOG_URL)
    dataset_names = client.dataset_names()
except Exception as err:
    raise RuntimeError(f"Cannot reach the catalog at {CATALOG_URL} — start it with `pixi run serve`.") from err
if DATASET_NAME not in dataset_names:
    raise RuntimeError(f"Dataset '{DATASET_NAME}' is missing — register episodes with `pixi run abc-register`.")

dataset = client.get_dataset(DATASET_NAME)
segment_ids = dataset.segment_ids()
print(f"Dataset '{DATASET_NAME}' holds {len(segment_ids)} segment(s)")

## Segment overview

Each episode is one segment; its recording properties surface as `property:…` columns of the segment table.

In [ ]:
tbl = dataset.segment_table().df.to_arrow_table()


def prop(name: str) -> list[str | None]:
    """Unwrap the single-element `property:` list column into plain values."""
    return [v[0] if v else None for v in tbl[f"property:{name}:{name}"].to_pylist()]


overview = pd.DataFrame({
    "segment": tbl["rerun_segment_id"].to_pylist(),
    "split": prop("split"),  # train/val
    "instruction": prop("instruction"),
    "station": prop("station"),  # e.g. RealSense, ZED, etc.
}).sort_values("segment", ignore_index=True)
overview

## Episodes with subtask annotation

Annotated episodes carry timestamped subtask labels at `/task/subtask` (a `StateChange` stream converted from `annotation.mcap`).
One reader across all segments is a single round-trip; each row carries its `rerun_segment_id`, so grouping happens locally.
The labels live on the `message_log_time` index — other indexes return no rows for this entity.

In [ ]:
SUBTASK_ENTITY = "/task/subtask"
STATE_COL = f"{SUBTASK_ENTITY}:StateChange:state"
INDEX = "message_log_time"

subtasks = dataset.filter_contents([SUBTASK_ENTITY]).reader(index=INDEX).to_arrow_table()
rows = [
    (seg, time, state[0])
    for seg, time, state in zip(
        subtasks["rerun_segment_id"].to_pylist(),
        subtasks[INDEX].to_pylist(),
        subtasks[STATE_COL].to_pylist(),
    )
    if state
]
rows.sort(key=lambda row: (row[0], row[1]))

annotated = sorted({seg for seg, _, _ in rows})
print(f"{len(annotated)} of {len(segment_ids)} episode(s) carry subtask annotations:")
for seg in annotated:
    print(f"  - {seg}")

Counter(label for _, _, label in rows).most_common()

## Subtasks containing 'mistake'

Imagine that you want to inspect what kind mistakes are made by teleoperation operators. The example below shows detecting segments that include `mistake` label(s) in the subtask annotation.


In [ ]:
window_end = {}
for (seg, time, _), (next_seg, next_time, _) in zip(rows, rows[1:]):
    if next_seg == seg:
        window_end[(seg, time)] = next_time

mistakes = pd.DataFrame(
    [
        {"segment": seg, "start": time, "end": window_end.get((seg, time)), "label": label}
        for seg, time, label in rows
        if "mistake" in label.lower()
    ],
    columns=["segment", "start", "end", "label"],
)

mistake_counts = mistakes.groupby("segment").size().rename("mistake_count")
print(f"{len(mistakes)} mistake label(s) across {mistake_counts.size} of {len(segment_ids)} episode(s)")
print(mistake_counts.to_string())
mistakes

## Inspect the episode with the most mistakes

Embed the Rerun viewer on that episode's catalog segment — seek to a `start` time from the table above to watch the mistake happen.

In [ ]:
from rerun.notebook import Viewer

if mistakes.empty:
    print("No mistake labels found.")
else:
    top_segment = mistake_counts.idxmax()
    print(f"Episode with the most mistakes: {top_segment} ({mistake_counts.max()} mistake(s))")
    Viewer(url=dataset.segment_url(top_segment), height=900, width=1200).display()